# Librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.use('TkAgg')

# Cargar el CSV

In [2]:
# Cargar el archivo CSV
df = pd.read_csv("retail_store_inventory.csv", sep=",", encoding="utf-8")

# Mostrar las primeras filas para verificar que se cargó correctamente
print(df.head())

         Date Store ID Product ID     Category Region  Inventory Level  \
0  2022-01-01     S001      P0001    Groceries  North              231   
1  2022-01-01     S001      P0002         Toys  South              204   
2  2022-01-01     S001      P0003         Toys   West              102   
3  2022-01-01     S001      P0004         Toys  North              469   
4  2022-01-01     S001      P0005  Electronics   East              166   

   Units Sold  Units Ordered  Demand Forecast  Price  Discount  \
0         127             55           135.47  33.50        20   
1         150             66           144.04  63.01        20   
2          65             51            74.02  27.99        10   
3          61            164            62.18  32.72        10   
4          14            135             9.26  73.64         0   

  Weather Condition  Holiday/Promotion  Competitor Pricing Seasonality  
0             Rainy                  0               29.69      Autumn  
1           

# Exploracion Inicial

In [14]:
# Estadísticas básicas
print("\n>>> Estadísticas básicas:")
print(df.describe(include="all"))

# Información del DataFrame
print("\n>>> Info del DataFrame:")
df.info()

# Número de filas y columnas
print("\n>>> Dimensiones (filas, columnas):")
print(df.shape)

# Valores nulos por columna
print("\n>>> Valores nulos por columna:")
print(df.isnull().sum())

# Verifica los nombres de todas las columnas
print(df.columns)


>>> Estadísticas básicas:
              Date Store ID Product ID   Category Region  Inventory Level  \
count        73100    73100      73100      73100  73100     73100.000000   
unique         731        5         20          5      4              NaN   
top     2022-01-01     S001      P0001  Furniture   East              NaN   
freq           100    14620       3655      14699  18349              NaN   
mean           NaN      NaN        NaN        NaN    NaN       274.469877   
std            NaN      NaN        NaN        NaN    NaN       129.949514   
min            NaN      NaN        NaN        NaN    NaN        50.000000   
25%            NaN      NaN        NaN        NaN    NaN       162.000000   
50%            NaN      NaN        NaN        NaN    NaN       273.000000   
75%            NaN      NaN        NaN        NaN    NaN       387.000000   
max            NaN      NaN        NaN        NaN    NaN       500.000000   

          Units Sold  Units Ordered  Demand Fore

# Normalizar columnas

In [3]:
# Convertir manualmente algunas columnas a categóricas
df["Category"] = df["Category"].astype("category")
df["Region"] = df["Region"].astype("category")

# Verificación de las columnas convertidas
print(df.dtypes)
# Normalizar las columnas categóricas
category_col = df.select_dtypes(include=["object"]).columns
print("\n>>> Columnas categóricas detectadas:")
print(list(category_col))

if len(category_col) > 0:
    df[category_col] = df[category_col].apply(lambda col: col.astype("string").str.strip().str.lower())
    print("\n>>> Ejemplo columnas categóricas normalizadas:")
    print(df[category_col].head())


Date                    object
Store ID                object
Product ID              object
Category              category
Region                category
Inventory Level          int64
Units Sold               int64
Units Ordered            int64
Demand Forecast        float64
Price                  float64
Discount                 int64
Weather Condition       object
Holiday/Promotion        int64
Competitor Pricing     float64
Seasonality             object
dtype: object

>>> Columnas categóricas detectadas:
['Date', 'Store ID', 'Product ID', 'Weather Condition', 'Seasonality']

>>> Ejemplo columnas categóricas normalizadas:
         Date Store ID Product ID Weather Condition Seasonality
0  2022-01-01     s001      p0001             rainy      autumn
1  2022-01-01     s001      p0002             sunny      autumn
2  2022-01-01     s001      p0003             sunny      summer
3  2022-01-01     s001      p0004            cloudy      autumn
4  2022-01-01     s001      p0005           

# Correlacion

In [4]:
# Normalizar las columnas categóricas
category_col = df.select_dtypes(include=["object"]).columns
print("\n>>> Columnas categóricas detectadas:")
print(list(category_col))

if len(category_col) > 0:
    df[category_col] = df[category_col].apply(lambda col: col.astype("string").str.strip().str.lower())
    print("\n>>> Ejemplo columnas categóricas normalizadas:")
    print(df[category_col].head())


>>> Columnas categóricas detectadas:
[]


# frecuencia por categoría

In [11]:
# Gráfico de barras de frecuencia para las categorías
if "Category" in df.columns:
    category_counts = df['Category'].value_counts()
    plt.figure(figsize=(10, 6))
    sns.barplot(x=category_counts.index, y=category_counts.values)
    plt.title("Frecuencia de productos por categoría")
    plt.xlabel("Categoría")
    plt.ylabel("Frecuencia")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Relación entre precio y unidades vendidas

In [13]:
plt.figure(figsize=(10, 6))
plt.hexbin(df['Price'], df['Units Sold'], gridsize=50, cmap='Blues')
plt.colorbar()
plt.title("Densidad entre Precio y Unidades Vendidas")
plt.xlabel("Precio")
plt.ylabel("Unidades Vendidas")
plt.show()

# Mapa de calor de ventas por región y clima

In [15]:
    # Mapa de calor de ventas por región y clima
if "Region" in df.columns and "Weather Condition" in df.columns:
    tabla = (
        df
        .groupby(["Region", "Weather Condition"])
        .size()
        .reset_index(name="conteo")
    )

    pivot = tabla.pivot(index="Region", columns="Weather Condition", values="conteo").fillna(0)

    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, cmap="magma")
    plt.title("Mapa de calor de ventas por región y condiciones climáticas")
    plt.xlabel("Condiciones Climáticas")
    plt.ylabel("Región")
    plt.tight_layout()
    plt.show()

C:\Users\isaac\AppData\Local\Temp\ipykernel_17316\482696323.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Region", "Weather Condition"])


# Correlacion numericas

In [14]:
# Seleccionar solo las columnas numéricas
df_num = df.select_dtypes(include=["number"])

# Calcular la matriz de correlación
corr_matrix = df_num.corr()

# Crear el mapa de calor
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Matriz de Correlación")
plt.tight_layout()
plt.show()